In [7]:
import os
import glob
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
 
BATCH_SIZE = 8
EPOCHS = 20
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")  # accélération Metal sur Mac Apple Silicon
else:
    device = torch.device("cpu")
print("Device :", device)

Device : mps


In [8]:
inv = pd.read_csv("inventaire_oasis2.csv")
 
clinical_path = glob.glob("*longitudinal*.xlsx") + glob.glob(
    os.path.join("..", "src", "Data", "*longitudinal*.xlsx"))
clinical = pd.read_excel(clinical_path[0])
print("Colonnes du fichier clinique :", clinical.columns.tolist())
print(clinical[["MRI ID", "Group", "Visit"]].head())

Colonnes du fichier clinique : ['Subject ID', 'MRI ID', 'Group', 'Visit', 'MR Delay', 'M/F', 'Hand', 'Age', 'EDUC', 'SES', 'MMSE', 'CDR', 'eTIV', 'nWBV', 'ASF']
          MRI ID        Group  Visit
0  OAS2_0001_MR1  Nondemented      1
1  OAS2_0001_MR2  Nondemented      2
2  OAS2_0002_MR1     Demented      1
3  OAS2_0002_MR2     Demented      2
4  OAS2_0002_MR3     Demented      3


In [9]:
first_visit = clinical[clinical["Visit"] == 1].copy()
first_visit = first_visit[first_visit["Group"].isin(["Nondemented", "Converted"])]
print(f"\n{len(first_visit)} sujets retenus (Nondemented + Converted, 1re visite).")
print(first_visit["Group"].value_counts())
 
first_visit = first_visit.rename(columns={"MRI ID": "subject_id"})
merged = first_visit.merge(inv, on="subject_id", how="inner")
merged = merged[merged["has_any_volume"]].reset_index(drop=True)
print(f"{len(merged)} sujets avec volume IRM disponible.")
 
label_map = {"Nondemented": 0, "Converted": 1}
merged["label"] = merged["Group"].map(label_map)


86 sujets retenus (Nondemented + Converted, 1re visite).
Group
Nondemented    72
Converted      14
Name: count, dtype: int64
86 sujets avec volume IRM disponible.


In [10]:
def load_central_slice(subject_path):
    img_files = sorted(glob.glob(os.path.join(subject_path, "**", "mpr-*.nifti.img"), recursive=True))
    volumes = [np.squeeze(nib.load(f).get_fdata()) for f in img_files]
    volume = np.mean(volumes, axis=0)
    return volume[:, :, volume.shape[2] // 2]
 
 
print("\nPréchargement des images en mémoire (une seule fois, évite de relire le "
      "disque à chaque epoch/fold)...")
cache = {}
for _, row in merged.iterrows():
    img = load_central_slice(row["path"])
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    img = torch.tensor(img, dtype=torch.float32).unsqueeze(0)
    img = torch.nn.functional.interpolate(img.unsqueeze(0), size=(128, 128)).squeeze(0)
    cache[row["subject_id"]] = img
print(f"{len(cache)} images préchargées.")
 
 
class OasisFirstVisitDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
 
    def __len__(self):
        return len(self.df)
 
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return cache[row["subject_id"]], row["label"]


Préchargement des images en mémoire (une seule fois, évite de relire le disque à chaque epoch/fold)...
86 images préchargées.


In [11]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 16 * 16, 64), nn.ReLU(), nn.Dropout(0.4),
            nn.Linear(64, 2),
        )
 
    def forward(self, x):
        return self.classifier(self.features(x))

In [12]:
X = merged.index.values
y = merged["label"].values
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
 
all_preds, all_labels, all_probs = [], [], []
 
for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    train_df = merged.iloc[train_idx]
    val_df = merged.iloc[val_idx]
 
    # pondération de classe pour compenser le déséquilibre Converted/Nondemented
    class_counts = train_df["label"].value_counts()
    weights = torch.tensor([1.0 / class_counts.get(0, 1), 1.0 / class_counts.get(1, 1)], dtype=torch.float32)
 
    train_loader = DataLoader(OasisFirstVisitDataset(train_df), batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(OasisFirstVisitDataset(val_df), batch_size=BATCH_SIZE)
 
    model = SimpleCNN().to(device)
    criterion = nn.CrossEntropyLoss(weight=weights.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
 
    for epoch in range(EPOCHS):
        model.train()
        for x, yb in train_loader:
            x, yb = x.to(device), yb.to(device)
            optimizer.zero_grad()
            loss = criterion(model(x), yb)
            loss.backward()
            optimizer.step()
 
    model.eval()
    with torch.no_grad():
        for x, yb in val_loader:
            x = x.to(device)
            probs = torch.softmax(model(x), dim=1).cpu().numpy()
            all_preds.extend(np.argmax(probs, axis=1))
            all_labels.extend(yb.numpy())
            all_probs.extend(probs[:, 1])
    print(f"Fold {fold + 1}/5 terminé.")

Fold 1/5 terminé.
Fold 2/5 terminé.
Fold 3/5 terminé.
Fold 4/5 terminé.
Fold 5/5 terminé.


In [13]:
print("\nRapport de classification (Nondemented=0, Converted=1) :")
print(classification_report(all_labels, all_preds, target_names=["Nondemented", "Converted"]))
print("Matrice de confusion :")
print(confusion_matrix(all_labels, all_preds))
print(f"AUC-ROC : {roc_auc_score(all_labels, all_probs):.3f}")
 
print("\nRappel : échantillon très petit (~14 Converted au total) -> résultats "
      "à interpréter avec prudence, à comparer avec un modèle classique "
      "(régression logistique) sur les biomarqueurs de la Partie 3.")


Rapport de classification (Nondemented=0, Converted=1) :
              precision    recall  f1-score   support

 Nondemented       0.84      0.89      0.86        72
   Converted       0.20      0.14      0.17        14

    accuracy                           0.77        86
   macro avg       0.52      0.52      0.52        86
weighted avg       0.74      0.77      0.75        86

Matrice de confusion :
[[64  8]
 [12  2]]
AUC-ROC : 0.533

Rappel : échantillon très petit (~14 Converted au total) -> résultats à interpréter avec prudence, à comparer avec un modèle classique (régression logistique) sur les biomarqueurs de la Partie 3.
